# Fundamentos de Delta Tables / Delta Lake

Este notebook explica qué es una tabla Delta, qué guarda en `_delta_log` y cuáles son los comandos más útiles para operación y optimización.

## Objetivos

Al finalizar, debes poder explicar y practicar:

1. Qué es Delta Lake / Delta Table.
2. Cómo crear una tabla Delta.
3. Qué se guarda en `_delta_log`.
4. Cómo usar `DESCRIBE DETAIL` y `DESCRIBE HISTORY`.
5. Cómo usar `OPTIMIZE`.
6. Cómo usar `VACUUM`.
7. Cómo usar Time Travel.
8. Cómo habilitar y leer Change Data Feed, conocido como CDF o CDC en algunos equipos.
9. Cómo calcular estadísticas con `ANALYZE TABLE ... COMPUTE STATISTICS`.

> Recomendación: ejecuta este notebook en Databricks. Algunos comandos como `OPTIMIZE`, `ZORDER`, `VACUUM` y CDF están pensados para Delta en Databricks Runtime.

## 1. Qué es Delta Lake

**Delta Lake** es un formato de almacenamiento abierto para Lakehouse. Técnicamente, una tabla Delta combina:

1. Archivos de datos en formato **Parquet**.
2. Un directorio llamado **`_delta_log`**.
3. Un protocolo transaccional que permite operaciones ACID.

En simple:

```text
Delta Table
├── part-0000...snappy.parquet       # datos físicos
├── part-0001...snappy.parquet       # datos físicos
└── _delta_log/
    ├── 00000000000000000000.json    # commit 0
    ├── 00000000000000000001.json    # commit 1
    └── ...
```

Delta no modifica los archivos Parquet directamente. En vez de eso, escribe nuevos archivos y registra en el log qué archivos están activos y cuáles fueron removidos lógicamente.

## 2. Crear una tabla Delta

Crearemos una tabla de eventos meteorológicos. En un proyecto Medallion, este tipo de tabla podría representar una capa Bronze o Silver, dependiendo del nivel de transformación aplicado.

## 3. Inspeccionar la tabla Delta

Comandos útiles:

- `DESCRIBE DETAIL`: metadatos físicos y lógicos de la tabla.
- `DESCRIBE HISTORY`: historial de commits.

## 4. Qué guarda `_delta_log`

El transaction log contiene acciones que describen el estado de la tabla.

Acciones comunes:

| Acción | Qué representa |
|---|---|
| `protocol` | Versiones mínimas de lectura/escritura requeridas |
| `metaData` | Schema, particiones, propiedades, ubicación |
| `add` | Archivo Parquet agregado a la versión activa |
| `remove` | Archivo removido lógicamente de la versión activa |
| `commitInfo` | Quién hizo la operación, cuándo, operación ejecutada, métricas |
| `txn` | Información de transacciones de aplicaciones, cuando aplica |

Importante: `remove` no borra inmediatamente el archivo físico. Lo marca como no activo. La eliminación física ocurre con `VACUUM`.

## 5. Updates, deletes y el transaction log

Cuando haces `UPDATE` o `DELETE`, Delta normalmente crea nuevos archivos Parquet y registra archivos removidos lógicamente en `_delta_log`.

## 6. Time Travel

Delta permite leer versiones anteriores de una tabla.

Esto es útil para:

- auditoría
- debugging
- comparar cambios
- reproducibilidad
- rollback lógico

Puedes viajar en el tiempo por versión o por timestamp.

In [0]:
# Ver el historial para identificar versiones disponibles.
history_df = spark.sql(f"DESCRIBE HISTORY {full_table_name}").select("version", "timestamp", "operation")
show_df(history_df)

In [0]:
# Leer versión 0 por path.
version_0_df = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
show_df(version_0_df.orderBy("event_id"))

# Leer versión actual.
current_df = spark.read.format("delta").load(delta_path)
show_df(current_df.orderBy("event_id"))

In [0]:
# También se puede consultar con SQL.
show_df(spark.sql(f"SELECT * FROM {full_table_name} VERSION AS OF 0 ORDER BY event_id"))

### Restaurar una tabla

Delta también permite restaurar una tabla a una versión anterior.

> Ejecuta este comando solo cuando realmente quieras revertir la tabla.

In [0]:
# Ejemplo opcional. Descomenta para restaurar a versión 0.
# spark.sql(f"RESTORE TABLE {full_table_name} TO VERSION AS OF 0")

## 7. `OPTIMIZE`

`OPTIMIZE` compacta archivos pequeños en archivos más grandes. Esto mejora el rendimiento de lectura, especialmente cuando tienes muchos archivos pequeños.

Casos típicos:

- Ingesta frecuente con micro-batches.
- Muchas particiones pequeñas.
- Tablas leídas frecuentemente por BI o dashboards.

Con Databricks también puedes usar `ZORDER BY` para mejorar data skipping en columnas muy consultadas.

In [0]:
# Compactación básica.
spark.sql(f"OPTIMIZE {full_table_name}")

# ZORDER es útil para columnas utilizadas frecuentemente en filtros.
# Ejemplo: location_id.
spark.sql(f"OPTIMIZE {full_table_name} ZORDER BY (location_id)")

show_df(spark.sql(f"DESCRIBE HISTORY {full_table_name}"))

## 8. `VACUUM`

`VACUUM` elimina físicamente archivos antiguos que ya no están activos en la tabla Delta.

Importante:

- Por defecto, Delta protege un retention mínimo de 7 días.
- Si haces `VACUUM`, puedes perder la capacidad de hacer Time Travel a versiones que dependen de archivos eliminados.
- No uses retention muy bajo en producción salvo que entiendas el impacto.

In [0]:
# Seguro: retener 7 días / 168 horas.
spark.sql(f"VACUUM {full_table_name} RETAIN 168 HOURS")

### Retention menor a 7 días

En ambientes de prueba se puede deshabilitar la validación de seguridad, pero **no es recomendado en producción**.

```python
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql(f"VACUUM {full_table_name} RETAIN 0 HOURS")
```

## 9. `ANALYZE TABLE ... COMPUTE STATISTICS`

El comando `ANALYZE TABLE` calcula estadísticas que pueden ayudar al optimizador de consultas.

Ejemplo pedido:

```sql
ANALYZE TABLE my_table COMPUTE STATISTICS;
```

También puedes calcular estadísticas para columnas específicas.

In [0]:
# Estadísticas de tabla.
spark.sql(f"ANALYZE TABLE {full_table_name} COMPUTE STATISTICS")

# Estadísticas de columnas.
spark.sql(f"ANALYZE TABLE {full_table_name} COMPUTE STATISTICS FOR COLUMNS location_id, weather_code, temperature_2m")

show_df(spark.sql(f"DESCRIBE EXTENDED {full_table_name}"))

## 10. Change Data Feed / CDC

En Delta, la funcionalidad se llama normalmente **Change Data Feed** o **CDF**.

Muchos equipos lo llaman CDC porque sirve para consumir cambios, pero técnicamente en Delta el feature es CDF.

CDF permite leer cambios como:

- `insert`
- `update_preimage`
- `update_postimage`
- `delete`

Columnas especiales:

- `_change_type`
- `_commit_version`
- `_commit_timestamp`

In [0]:
# Habilitar Change Data Feed.
spark.sql(f"ALTER TABLE {full_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Capturamos la versión actual para leer cambios desde la siguiente versión.
start_version = spark.sql(f"DESCRIBE HISTORY {full_table_name}").agg(F.max("version").alias("max_version")).collect()[0]["max_version"]
print(f"CDF se leerá desde la versión: {start_version + 1}")

In [0]:
# Generamos cambios después de habilitar CDF.
spark.sql(f"INSERT INTO {full_table_name} VALUES ('e005', 'PTY', timestamp('2026-06-26 10:00:00'), 30.4, 0.0, 12.1, 2)")

spark.sql(f"UPDATE {full_table_name} SET rain = 1.5 WHERE event_id = 'e002'")

spark.sql(f"DELETE FROM {full_table_name} WHERE event_id = 'e003'")

show_df(spark.table(full_table_name).orderBy("event_id"))

In [0]:
# Leer cambios usando Change Data Feed.
cdf_df = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", start_version + 1)
    .load(delta_path)
)

show_df(
    cdf_df.select(
        "_commit_version",
        "_commit_timestamp",
        "_change_type",
        "event_id",
        "location_id",
        "event_ts",
        "temperature_2m",
        "rain"
    ).orderBy("_commit_version", "event_id", "_change_type")
)

## 11. Comandos Delta más útiles

| Comando | Para qué sirve |
|---|---|
| `DESCRIBE DETAIL table` | Ver metadatos físicos/lógicos de la tabla |
| `DESCRIBE HISTORY table` | Ver historial de commits |
| `OPTIMIZE table` | Compactar archivos pequeños |
| `OPTIMIZE table ZORDER BY (col)` | Mejorar data skipping por columnas filtradas frecuentemente |
| `VACUUM table RETAIN 168 HOURS` | Borrar archivos físicos antiguos no activos |
| `SELECT * FROM table VERSION AS OF n` | Time Travel por versión |
| `RESTORE TABLE table TO VERSION AS OF n` | Restaurar una versión anterior |
| `ALTER TABLE table SET TBLPROPERTIES (...)` | Configurar propiedades Delta |
| `ANALYZE TABLE table COMPUTE STATISTICS` | Calcular estadísticas para el optimizador |
| `MERGE INTO` | Upsert, SCD1, SCD2, deduplicación avanzada |

## 12. Recomendaciones operativas

| Tema | Recomendación |
|---|---|
| Archivos pequeños | Usar `OPTIMIZE` en tablas con muchas escrituras pequeñas |
| Time Travel | Definir política de retención según auditoría y costos |
| VACUUM | No usar retention bajo sin aprobación técnica |
| CDF | Habilitar antes de necesitar cambios; no captura cambios históricos anteriores |
| Estadísticas | Usar `ANALYZE TABLE` en tablas consultadas por BI o joins grandes |
| Particionamiento | Particionar solo por columnas de baja/mediana cardinalidad y filtros frecuentes |
| ZORDER | Útil para columnas de alta cardinalidad filtradas frecuentemente |
| Governance | Usar Unity Catalog para permisos, lineage y auditoría |

## Resumen del notebook

Delta Tables resuelve varios problemas clásicos de Data Lakes:

- Falta de transacciones.
- Dificultad para hacer updates/deletes/merges.
- Falta de historial confiable.
- Problemas de archivos pequeños.
- Necesidad de auditoría y reproducibilidad.

La idea central:

```text
Parquet = almacenamiento eficiente
_delta_log = historial + transacciones + estado de la tabla
Delta Table = Parquet + transaction log + comandos Lakehouse
```